# Phase 2: Data cleaning and data-quality audit

**CSE558 Data Science midterm project · Olist Brazilian e-commerce dataset**

This notebook audits the Phase 1 analysis table and the source CSVs. It makes only justified changes, and it records every decision in a cleaning log. It does **no EDA and no hypothesis testing**.

**What this notebook does not do.** It does not repeat the Phase 1 filters. It also does not change the review rule, the lateness rule or the review-timing definition.

**What it does to the sample.** It applies **one chronological consistency exclusion** (section 7): orders whose review was answered before the order was purchased. This is an extra funnel step after the Phase 1 rules.

**What it produces.** All outputs go to `outputs/phase2/`, and the Phase 1 outputs in `outputs/` are left untouched:

- `analysis_table_audited` (95,299 rows): every Phase 1 order, with all quality flags plus `excluded_from_primary` and `exclusion_reason`, so the exclusion is traceable and reversible.
- `analysis_table_final`, `outcomes_final` and `X_encoded_final`: the **final primary sample**, all in the same `order_id` order.
- The cleaning log, missingness summary, date and value checks, encoding checks and a complete feature dictionary.

**How to run.** Run `01_build_analysis_table.ipynb` first, then this notebook with *Kernel → Restart & Run All*. Both need the nine Olist CSVs in `data/`; see the README.

In [1]:
import os
from pathlib import Path
# All paths are relative to the project root. If Jupyter starts inside notebooks/, move up one level.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
import hashlib
import json

import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

DATA_DIR = Path("data")
P1_DIR = Path("outputs") / "phase1"  # Phase 1 outputs (inputs here)
OUT2 = Path("outputs") / "phase2"    # everything Phase 2 writes
OUT2.mkdir(parents=True, exist_ok=True)

# 1. The source data must be exactly the files Phase 1 used: compare with the manifest
manifest = pd.read_csv("data_manifest.csv")
check = []
for _, m in manifest.iterrows():
    f = DATA_DIR / m.file
    assert f.exists(), f"missing source file: {f} (see README: data folder layout)"
    sha = hashlib.sha256(f.read_bytes()).hexdigest()
    check.append({"file": m.file, "exists": True, "sha256_matches": sha == m.sha256})
check = pd.DataFrame(check)
assert check.sha256_matches.all(), "a source CSV differs from the version Phase 1 was built on"

# 2. Phase 1 outputs must exist
for f in ["analysis_table.parquet", "outcomes.csv", "X_encoded.parquet", "02_filter_funnel.csv",
          "03_quality_flags.csv", "04_feature_dictionary.csv", "05_phase1_summary.json"]:
    assert (P1_DIR / f).exists(), f"run 01_build_analysis_table.ipynb first: {f} not found"
print("Source CSVs match data_manifest.csv; Phase 1 outputs found.")
check

Source CSVs match data_manifest.csv; Phase 1 outputs found.


,file,exists,sha256_matches
0,olist_customers_dataset.csv,True,True
1,olist_geolocation_dataset.csv,True,True
2,olist_order_items_dataset.csv,True,True
3,olist_order_payments_dataset.csv,True,True
4,olist_order_reviews_dataset.csv,True,True
5,olist_orders_dataset.csv,True,True
6,olist_products_dataset.csv,True,True
7,olist_sellers_dataset.csv,True,True
8,product_category_name_translation.csv,True,True


## 1. Baseline and verification of Phase 1 decisions

Before auditing anything, we load what Phase 1 produced and verify the Phase 1 integration rules against the files themselves.

In [2]:
a = pd.read_parquet(P1_DIR / "analysis_table.parquet")
outcomes = pd.read_csv(P1_DIR / "outcomes.csv", index_col=0)
X = pd.read_parquet(P1_DIR / "X_encoded.parquet")
funnel = pd.read_csv(P1_DIR / "02_filter_funnel.csv")
p1_flags = pd.read_csv(P1_DIR / "03_quality_flags.csv", index_col=0)
p1_summary = json.load(open(P1_DIR / "05_phase1_summary.json"))

src = {n: pd.read_csv(DATA_DIR / f) for n, f in {
    "orders": "olist_orders_dataset.csv", "reviews": "olist_order_reviews_dataset.csv",
    "items": "olist_order_items_dataset.csv", "payments": "olist_order_payments_dataset.csv",
    "customers": "olist_customers_dataset.csv", "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv", "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv"}.items()}
n_rev = src["reviews"].order_id.value_counts()

decisions = {
    "one row per order_id": a.order_id.is_unique,
    "rows equal Phase 1 summary": len(a) == p1_summary["eligible_orders"],
    "all orders have status 'delivered'": (a.order_status == "delivered").all(),
    "all orders have an actual delivery timestamp": a.order_delivered_customer_date.notna().all(),
    "every order has exactly one review in the source": (n_rev.reindex(a.order_id) == 1).all(),
    "is_late = delivery calendar date > estimated date": (
        a.is_late == (a.order_delivered_customer_date.dt.normalize() > a.order_estimated_delivery_date)).all(),
    "review_before_delivery = answer timestamp < delivery timestamp": (
        a.review_before_delivery == (a.review_answer_timestamp < a.order_delivered_customer_date)).all(),
    "shared-review flag present and kept": "review_shared_across_orders" in a and a.review_shared_across_orders.sum() > 0,
    "analysis table keeps original missingness (weights not imputed)": a.weight_total_g.isna().sum() > 0,
    "encoded matrix has no missing values": not X.isna().any().any(),
    "same order_id, same order in table, outcomes and X": (
        (a.order_id.values == outcomes.index.values).all() and (a.order_id.values == X.index.values).all()),
}
decisions = pd.Series(decisions, name="verified").to_frame()
assert decisions.verified.all(), decisions[~decisions.verified]
decisions

,verified
one row per order_id,True
rows equal Phase 1 summary,True
all orders have status 'delivered',True
all orders have an actual delivery timestamp,True
every order has exactly one review in the source,True
is_late = delivery calendar date > estimated date,True
review_before_delivery = answer timestamp < delivery timestamp,True
shared-review flag present and kept,True
analysis table keeps original missingness (weights not imputed),True
encoded matrix has no missing values,True


In [3]:
baseline = pd.DataFrame([
    {"item": "orders in source", "value": len(src["orders"])},
    {"item": "rows in analysis table", "value": len(a)},
    {"item": "columns in analysis table", "value": a.shape[1]},
    {"item": "late orders (calendar day)", "value": int(a.is_late.sum())},
    {"item": "missing cells in analysis table", "value": int(a.isna().sum().sum())},
    {"item": "encoded matrix rows x columns", "value": f"{X.shape[0]} x {X.shape[1]}"},
    {"item": "rows removed by Phase 1 filters", "value": int(funnel.removed.sum())},
])
baseline.to_csv(OUT2 / "p2_baseline.csv", index=False)
display(baseline)
print("Phase 1 quality flags (baseline):")
p1_flags

,item,value
0,orders in source,99441
1,rows in analysis table,95299
2,columns in analysis table,55
3,late orders (calendar day),6353
4,missing cells in analysis table,140755
5,encoded matrix rows x columns,95299 x 167
6,rows removed by Phase 1 filters,4142


Phase 1 quality flags (baseline):


,orders
wait_days <= 0,0
wait_days > 60,290
promised_days <= 0,0
delay over 30 days late,341
orders with a zero-weight product,6
payment_total != price + freight (by > 1 BRL),245
orders with no payment record,1
comment present but only spaces (counted as no comment),26
delay_days > 0 but is_late = 0 (promised day),1269
review answered before delivery (D4),4627


The Phase 1 funnel already removed every order that fails the Phase 1 eligibility rules. **This notebook applies no further row filters**, so no observation can be excluded twice.

## 2. Missing values

We audit every column of every source table and of the analysis table. The proposed treatment is given for each column with missing values. A column with no missing values needs none.

In [4]:
TREAT = {
    ("orders", "order_approved_at"): ("kept missing", "approval time is not used by any feature"),
    ("orders", "order_delivered_carrier_date"): ("kept missing", "carrier hand-over time is not used by any feature"),
    ("orders", "order_delivered_customer_date"): ("orders excluded (Phase 1 funnel)", "without the arrival time, lateness and waiting time cannot be computed; nearly all are not-delivered statuses"),
    ("reviews", "review_comment_title"): ("kept missing; has_title = 0", "an absent title is information, not an error; text left unchanged"),
    ("reviews", "review_comment_message"): ("kept missing; has_comment = 0", "customers may rate without writing; text left unchanged for Phase 3"),
    ("products", "product_category_name"): ("explicit 'unknown' category at order level", "no category is invented; cat__unknown and flag_category_unknown mark these orders"),
    ("products", "product_name_lenght"): ("kept missing", "not used by any feature"),
    ("products", "product_description_lenght"): ("kept missing", "not used by any feature"),
    ("products", "product_photos_qty"): ("kept missing", "not used by any feature"),
    ("products", "product_weight_g"): ("order total left missing", "summing with a gap would understate weight; median used only in the encoded matrix"),
    ("products", "product_length_cm"): ("order volume left missing", "same reason as weight"),
    ("products", "product_height_cm"): ("order volume left missing", "same reason as weight"),
    ("products", "product_width_cm"): ("order volume left missing", "same reason as weight"),
    ("analysis", "order_approved_at"): ("kept missing", "not used by any feature"),
    ("analysis", "order_delivered_carrier_date"): ("kept missing", "not used by any feature"),
    ("analysis", "review_comment_title"): ("kept missing; has_title = 0", "text left unchanged"),
    ("analysis", "review_comment_message"): ("kept missing; has_comment = 0", "text left unchanged for Phase 3"),
    ("analysis", "weight_total_g"): ("missing in table; median in X only", "an order with an item of unknown weight has unknown total weight"),
    ("analysis", "volume_total_cm3"): ("missing in table; median in X only", "same reason as weight"),
    ("analysis", "payment_total"): ("missing in table; median in X only", "one order has no payment record in the source"),
    ("analysis", "n_payment_records"): ("missing in table; median in X only", "same order as payment_total"),
    ("analysis", "max_installments"): ("missing in table; median in X only", "same order as payment_total"),
    ("analysis", "payment_types"): ("kept missing; all pay__ indicators 0 in X", "same order as payment_total"),
}

rows = []
tables = dict(src)
tables["analysis"] = a
for tname, d in tables.items():
    for c in d.columns:
        n = int(d[c].isna().sum())
        treat, why = TREAT.get((tname, c), ("none needed", "no missing values") if n == 0 else ("REVIEW", "no treatment defined"))
        rows.append({"table": tname, "column": c, "dtype": str(d[c].dtype), "rows": len(d), "missing": n,
                     "missing_pct": round(100 * n / len(d), 2), "treatment": treat, "justification": why})
missingness = pd.DataFrame(rows)
assert (missingness.treatment != "REVIEW").all(), missingness[missingness.treatment == "REVIEW"]
missingness.to_csv(OUT2 / "p2_missingness_summary.csv", index=False)

print("Known issues verified from source:",
      f"products without category = {src['products'].product_category_name.isna().sum():,};",
      f"orders without delivery date = {src['orders'].order_delivered_customer_date.isna().sum():,}")
missingness[missingness.missing > 0]

Known issues verified from source: products without category = 610; orders without delivery date = 2,965


,table,column,dtype,rows,missing,missing_pct,treatment,justification
4,orders,order_approved_at,str,99441,160,0.16,kept missing,approval time is not used by any feature
5,orders,order_delivered_carrier_date,str,99441,1783,1.79,kept missing,carrier hand-over time is not used by any feature
6,orders,order_delivered_customer_date,str,99441,2965,2.98,orders excluded (Phase 1 funnel),"without the arrival time, lateness and waiting time cannot be computed; near..."
11,reviews,review_comment_title,str,99224,87656,88.34,kept missing; has_title = 0,"an absent title is information, not an error; text left unchanged"
12,reviews,review_comment_message,str,99224,58247,58.70,kept missing; has_comment = 0,customers may rate without writing; text left unchanged for Phase 3
33,products,product_category_name,str,32951,610,1.85,explicit 'unknown' category at order level,no category is invented; cat__unknown and flag_category_unknown mark these o...
34,products,product_name_lenght,float64,32951,610,1.85,kept missing,not used by any feature
35,products,product_description_lenght,float64,32951,610,1.85,kept missing,not used by any feature
36,products,product_photos_qty,float64,32951,610,1.85,kept missing,not used by any feature
37,products,product_weight_g,float64,32951,2,0.01,order total left missing,summing with a gap would understate weight; median used only in the encoded ...


### Review text: null, empty, whitespace-only and real comments

pandas turns an empty CSV field into a missing value by default. To tell the four cases apart, the raw review file is read again with that conversion switched off. It also turns words such as "NA" or "null" into missing values, so we check that no real comment was lost that way.

In [5]:
raw_rev = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv", keep_default_na=False, na_values=[])
raw_rev = raw_rev[raw_rev.order_id.isin(a.order_id)]
assert len(raw_rev) == len(a)

file_text = (DATA_DIR / "olist_order_reviews_dataset.csv").read_text(encoding="utf-8")
print('Quoted empty fields ("") in the file:', file_text.count(',"",'),
      '-> an empty field is the only way "no text" is stored, so null and empty cannot differ in this file')

NA_WORDS = ["NA", "N/A", "n/a", "NaN", "nan", "null", "NULL", "None", "#N/A", "<NA>"]
text_rows = []
for col in ["review_comment_message", "review_comment_title"]:
    s = raw_rev[col]
    cats = {"null or empty field": (s == "").sum(),
            "whitespace only": ((s != "") & (s.str.strip() == "")).sum(),
            "non-empty text": (s.str.strip() != "").sum()}
    for k, v in cats.items():
        text_rows.append({"column": col, "category": k, "orders": int(v), "pct_of_eligible": round(100 * v / len(s), 2)})
    print(f"{col}: values that default NA parsing would have wiped: {int(s.isin(NA_WORDS).sum())}")
text_cats = pd.DataFrame(text_rows)
text_cats.to_csv(OUT2 / "p2_comment_text_categories.csv", index=False)

no_comment_eligible = 100 * (a.has_comment == 0).mean()
no_comment_source = 100 * src["reviews"].review_comment_message.isna().mean()
title_only = int(((a.has_title == 1) & (a.has_comment == 0)).sum())
print(f"Without a comment: {no_comment_eligible:.2f}% of the {len(a):,} eligible orders "
      f"(the often-quoted {no_comment_source:.1f}% is for all {len(src['reviews']):,} review rows)")
print(f"Title but no message: {title_only:,} orders (has_comment = 0, but the customer did write something)")
assert (a.has_comment == (raw_rev.set_index('order_id').review_comment_message.reindex(a.order_id).str.strip() != "").values).all()
text_cats

Quoted empty fields ("") in the file: 0 -> an empty field is the only way "no text" is stored, so null and empty cannot differ in this file
review_comment_message: values that default NA parsing would have wiped: 0
review_comment_title: values that default NA parsing would have wiped: 0
Without a comment: 59.39% of the 95,299 eligible orders (the often-quoted 58.7% is for all 99,224 review rows)
Title but no message: 1,702 orders (has_comment = 0, but the customer did write something)


,column,category,orders,pct_of_eligible
0,review_comment_message,null or empty field,56575,59.37
1,review_comment_message,whitespace only,26,0.03
2,review_comment_message,non-empty text,38698,40.61
3,review_comment_title,null or empty field,84129,88.28
4,review_comment_title,whitespace only,2,0.00
5,review_comment_title,non-empty text,11168,11.72


**Decisions on review text.**

- Comment text is **not changed**.
- `has_comment` counts only a non-blank **message**, as Phase 1 defined it.
- "Title only" reviews (counted above) are a nuance worth flagging: the customer wrote a title but no message. A new derived column, `has_any_text`, is 1 when either field has text. Phase 4 can then choose which definition of "wrote a comment" it tests, and say so.

## 3. Dates, durations and impossible values

In [6]:
o_raw = src["orders"]
parse_rows = []
for c in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
          "order_delivered_customer_date", "order_estimated_delivery_date"]:
    p = pd.to_datetime(o_raw[c], errors="coerce", format="%Y-%m-%d %H:%M:%S")
    parse_rows.append({"table": "orders", "column": c, "non_missing": int(o_raw[c].notna().sum()),
                       "unparseable": int((p.isna() & o_raw[c].notna()).sum())})
for c in ["review_creation_date", "review_answer_timestamp"]:
    p = pd.to_datetime(src["reviews"][c], errors="coerce", format="%Y-%m-%d %H:%M:%S")
    parse_rows.append({"table": "reviews", "column": c, "non_missing": int(src["reviews"][c].notna().sum()),
                       "unparseable": int((p.isna() & src["reviews"][c].notna()).sum())})
parse_check = pd.DataFrame(parse_rows)
assert parse_check.unparseable.sum() == 0
parse_check

,table,column,non_missing,unparseable
0,orders,order_purchase_timestamp,99441,0
1,orders,order_approved_at,99281,0
2,orders,order_delivered_carrier_date,97658,0
3,orders,order_delivered_customer_date,96476,0
4,orders,order_estimated_delivery_date,99441,0
5,reviews,review_creation_date,99224,0
6,reviews,review_answer_timestamp,99224,0


In [7]:
P, A, C = a.order_purchase_timestamp, a.order_approved_at, a.order_delivered_carrier_date
D, E = a.order_delivered_customer_date, a.order_estimated_delivery_date
RC, RA = a.review_creation_date, a.review_answer_timestamp
hours = lambda s: s.dt.total_seconds() / 3600

def rng(series):
    s = series.dropna()
    return "" if s.empty else f"{s.min():.1f} to {s.max():.1f}"

date_checks = []
def dc(check, mask, values, unit, explanation, handling, used):
    date_checks.append({"check": check, "orders": int(mask.sum()), "range_of_values": rng(values[mask]) if mask.any() else "",
                        "unit": unit, "likely_explanation": explanation, "handling": handling, "variable_used_in_analysis": used})

dc("delivered before purchase", D < P, hours(D - P), "hours", "would be impossible", "none found", "yes")
dc("estimated date before purchase", E < P, hours(E - P), "hours", "would be impossible", "none found", "yes")
dc("approved before purchase", A < P, hours(A - P), "hours", "would be impossible", "none found", "no")
dc("handed to carrier before purchase", C < P, hours(P - C), "hours before purchase", "impossible order; most likely a logging error in the carrier timestamp", "flagged (flag_lifecycle_inconsistent); column unused", "no")
dc("customer delivery before carrier hand-over", D < C, hours(C - D), "hours", "impossible order; most likely the carrier scan was logged late", "flagged (flag_lifecycle_inconsistent); column unused", "no")
dc("handed to carrier before approval", C < A, hours(A - C), "hours", "approval logged after dispatch; possible for some payment flows, cause not recorded", "retained; plausible process order, column unused", "no")
dc("survey created before purchase", RC.dt.normalize() < P.dt.normalize(), hours(P - RC), "hours", "review probably refers to another purchase (two of three share a review_id)", "excluded from the primary sample (section 7)", "yes (outcome)")
dc("review answered before survey created", RA < RC, hours(RC - RA), "hours", "would be impossible", "none found", "yes")
dc("review answered before purchase", RA < P, hours(P - RA), "hours", "same three orders as above", "excluded from the primary sample (section 7)", "yes (outcome)")
dc("review answered before actual delivery", RA < D, hours(D - RA), "hours", "for late orders the survey is typically created about 2 days after the promised date, before arrival (Phase 1)", "retained and flagged (review_before_delivery, D4)", "yes")
dc("waiting time zero or negative", a.wait_days <= 0, a.wait_days, "days", "would be impossible", "none found", "yes")
dc("waiting time under 1 day", a.wait_days < 1, a.wait_days, "days", "very fast delivery; plausible for nearby sellers", "retained", "yes")
dc("waiting time over 60 days", a.wait_days > 60, a.wait_days, "days", "long but possible deliveries; no evidence in the data that they are errors", "retained and flagged (flag_wait_over_60)", "yes")
dc("more than 30 days late", a.delay_days > 30, a.delay_days, "days", "severe but possible delays; no evidence of error", "retained and flagged (flag_delay_over_30)", "yes")
dc("more than 30 days early", a.delay_days < -30, a.delay_days, "days", "Olist pads its estimates; early arrival is common", "retained", "yes")
dc("promised in under 3 days", a.promised_days < 3, a.promised_days, "days", "fast local shipping", "retained", "yes")
dc("promised over 60 days", a.promised_days > 60, a.promised_days, "days", "long promised times; cause not recorded", "retained", "yes")
dc("delay_days > 0 but is_late = 0", (a.delay_days > 0) & (a.is_late == 0), a.delay_days, "days",
   "delivered later in the day on the promised date (estimate stored as midnight)", "correct under D3; flagged (flag_promised_day_positive_delay)", "yes")
dc("answered over 60 days after delivery", a.review_lag_days > 60, a.review_lag_days, "days", "customer answered long after delivery; cause not recorded", "retained and flagged (flag_answered_over_60)", "yes")
dc("approval timestamp missing", A.isna(), pd.Series(np.nan, index=a.index), "", "not recorded", "kept missing; unused", "no")
dc("carrier timestamp missing", C.isna(), pd.Series(np.nan, index=a.index), "", "not recorded", "kept missing; unused", "no")

date_checks = pd.DataFrame(date_checks)
date_checks.to_csv(OUT2 / "p2_date_checks.csv", index=False)
date_checks

,check,orders,range_of_values,unit,likely_explanation,handling,variable_used_in_analysis
0,delivered before purchase,0,,hours,would be impossible,none found,yes
1,estimated date before purchase,0,,hours,would be impossible,none found,yes
2,approved before purchase,0,,hours,would be impossible,none found,no
3,handed to carrier before purchase,164,0.0 to 4109.1,hours before purchase,impossible order; most likely a logging error in the carrier timestamp,flagged (flag_lifecycle_inconsistent); column unused,no
4,customer delivery before carrier hand-over,23,0.4 to 386.3,hours,impossible order; most likely the carrier scan was logged late,flagged (flag_lifecycle_inconsistent); column unused,no
5,handed to carrier before approval,1341,0.0 to 4109.3,hours,"approval logged after dispatch; possible for some payment flows, cause not r...","retained; plausible process order, column unused",no
6,survey created before purchase,3,115.4 to 878.9,hours,review probably refers to another purchase (two of three share a review_id),excluded from the primary sample (section 7),yes (outcome)
7,review answered before survey created,0,,hours,would be impossible,none found,yes
8,review answered before purchase,3,41.3 to 788.6,hours,same three orders as above,excluded from the primary sample (section 7),yes (outcome)
9,review answered before actual delivery,4627,0.0 to 4482.3,hours,for late orders the survey is typically created about 2 days after the promi...,"retained and flagged (review_before_delivery, D4)",yes


In [8]:
# The three reviews dated before their order was purchased
before_purchase = a.review_answer_timestamp < a.order_purchase_timestamp
bp = a.loc[before_purchase, ["order_id", "review_id", "order_purchase_timestamp", "review_creation_date",
                             "review_answer_timestamp", "review_shared_across_orders", "is_late", "review_score"]]
shared_with = {rid: src["reviews"].loc[src["reviews"].review_id == rid, "order_id"].tolist() for rid in bp.review_id}
bp["orders_sharing_this_review"] = bp.review_id.map(lambda r: len(shared_with[r]))
bp.to_csv(OUT2 / "p2_reviews_before_purchase.csv", index=False)
bp

,order_id,review_id,order_purchase_timestamp,review_creation_date,review_answer_timestamp,review_shared_across_orders,is_late,review_score,orders_sharing_this_review
31108,ebc94658c583ab37ad4f8e9091c4bef2,d25026282e14fc531f0711246e121cc2,2018-04-14 08:22:43,2018-03-30,2018-04-02 00:32:35,1,0,2,2
47851,4bb9c2002502ca416276dc1ff5efb1b3,0857872f03ef22e3dbe8391c4dd331f6,2018-06-23 14:53:03,2018-05-18,2018-05-21 18:15:06,0,0,1,1
66814,4a62fb19d5fa08fb514619dfcc617b3d,a935b3d67f406bac4853776d80a05220,2018-05-09 19:23:46,2018-05-05,2018-05-08 02:06:50,1,0,3,2


**Chronological consistency exclusion (applied in section 7).** A review answered before the order was even placed cannot be a reaction to that order, because there was nothing yet to rate. Two of the three reviews also carry a review_id shared with another order. That strengthens the concern that the rating belongs to a different purchase, but the timestamps alone cannot prove *which* purchase it refers to. We do not need to know that: it is enough that the rating cannot be about this order.

**Time coverage** is uneven at the edges, which matters for Phase 3 time plots:

In [9]:
by_month = a.groupby(a.order_purchase_timestamp.dt.to_period("M")).size()
coverage = pd.DataFrame({"orders": by_month})
coverage.index = coverage.index.astype(str)
print(f"2016 has only {int((a.purchase_year == 2016).sum())} orders, in months:",
      list(coverage.index[coverage.index.str.startswith('2016')]))
print("Missing months between first and last purchase:",
      sorted(set(pd.period_range(by_month.index.min(), by_month.index.max(), freq="M").astype(str)) - set(coverage.index)))
coverage.T

2016 has only 262 orders, in months: ['2016-09', '2016-10', '2016-12']
Missing months between first and last purchase: ['2016-11']


order_purchase_timestamp,2016-09,2016-10,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,2017-09,2017-10,2017-11,2017-12,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08
orders,1,260,1,735,1635,2512,2284,3481,3080,3803,4123,4092,4418,7185,5435,6973,6424,6911,6741,6714,6066,6097,6328


## 4. Extreme values and inconsistent totals

Distributions are summarised, and nothing is trimmed or winsorised. The "far-out" count uses Tukey's 3 × IQR fence and is **descriptive only**: it says how heavy the tail is, not which values are wrong.

In [10]:
NUM = ["items_price_total", "freight_total", "freight_ratio", "weight_total_g", "volume_total_cm3", "n_items",
       "n_products", "n_sellers", "n_categories", "payment_total", "n_payment_records", "max_installments",
       "delay_days", "wait_days", "promised_days", "review_lag_days", "comment_len_chars"]
q = a[NUM].quantile([0, 0.01, 0.25, 0.5, 0.75, 0.99, 1]).T
q.columns = ["min", "p1", "q1", "median", "q3", "p99", "max"]
iqr = q.q3 - q.q1
q["far_out_count_3IQR"] = [int(((a[c] < q.q1[c] - 3 * iqr[c]) | (a[c] > q.q3[c] + 3 * iqr[c])).sum()) for c in NUM]
q["missing"] = a[NUM].isna().sum()
q["handling"] = "retained, no trimming"
q = q.round(2)
q.to_csv(OUT2 / "p2_numeric_distributions.csv")
q

,min,p1,q1,median,q3,p99,max,far_out_count_3IQR,missing,handling
items_price_total,0.85,11.99,45.90,86.70,149.90,990.00,13440.00,3883,0,"retained, no trimming"
freight_total,0.00,7.39,13.84,17.17,23.99,104.16,1794.96,4833,0,"retained, no trimming"
freight_ratio,0.00,0.03,0.13,0.22,0.38,1.46,21.45,2130,0,"retained, no trimming"
weight_total_g,0.00,99.00,300.00,750.00,2050.00,22350.00,184400.00,8302,16,"retained, no trimming"
volume_total_cm3,168.00,352.00,2964.00,7220.00,19800.00,135303.52,1476000.00,4032,16,"retained, no trimming"
n_items,1.00,1.00,1.00,1.00,1.00,3.00,21.00,9455,0,"retained, no trimming"
n_products,1.00,1.00,1.00,1.00,1.00,2.00,8.00,3121,0,"retained, no trimming"
n_sellers,1.00,1.00,1.00,1.00,1.00,2.00,5.00,1248,0,"retained, no trimming"
n_categories,1.00,1.00,1.00,1.00,1.00,1.00,3.00,765,0,"retained, no trimming"
payment_total,9.59,22.38,61.88,105.28,176.23,1048.72,13664.08,3790,1,"retained, no trimming"


In [11]:
items = src["items"][src["items"].order_id.isin(a.order_id)]
pay = src["payments"][src["payments"].order_id.isin(a.order_id)]
products = src["products"]

t = a.set_index("order_id")
diff = t.payment_total - t.items_price_total - t.freight_total
mismatch = diff.abs() > 1
seq = pay.groupby("order_id").payment_sequential.agg(["max", "size"])
seq_gap = (seq["max"] != seq["size"]).reindex(t.index, fill_value=False)
has_voucher = pay.assign(v=pay.payment_type == "voucher").groupby("order_id").v.any().reindex(t.index, fill_value=False)
over = mismatch & (diff > 0)
under = mismatch & (diff < 0)

payment_profile = pd.DataFrame([
    {"measure": "orders with |payment - (price + freight)| > 1 BRL", "value": int(mismatch.sum())},
    {"measure": "  of which payment is HIGHER", "value": int(over.sum())},
    {"measure": "  of which payment is LOWER", "value": int(under.sum())},
    {"measure": "median absolute difference (BRL)", "value": round(float(diff[mismatch].abs().median()), 2)},
    {"measure": "median difference as share of price + freight", "value": round(float((diff[mismatch].abs() / (t.items_price_total + t.freight_total)[mismatch]).median()), 3)},
    {"measure": "share of HIGHER-payment orders paid in > 1 instalment", "value": round(float((t.max_installments[over] > 1).mean()), 3)},
    {"measure": "share of ALL orders paid in > 1 instalment", "value": round(float((t.max_installments > 1).mean()), 3)},
    {"measure": "share of mismatches that used a voucher", "value": round(float(has_voucher[mismatch].mean()), 3)},
    {"measure": "orders with a gap in payment_sequential numbering", "value": int(seq_gap.sum())},
    {"measure": "  of which are mismatches", "value": int((seq_gap & mismatch).sum())},
    {"measure": "orders with no payment record", "value": int(t.payment_missing.sum())},
    {"measure": "payment records with 0 instalments", "value": int((pay.payment_installments == 0).sum())},
    {"measure": "payment records with value 0 (all vouchers?)", "value": int((pay.payment_value == 0).sum())},
    {"measure": "  payment types of zero-value records", "value": str(pay.loc[pay.payment_value == 0, "payment_type"].value_counts().to_dict())},
])
payment_profile.to_csv(OUT2 / "p2_payment_mismatch_profile.csv", index=False)
payment_profile

,measure,value
0,orders with |payment - (price + freight)| > 1 BRL,245
1,of which payment is HIGHER,227
2,of which payment is LOWER,18
3,median absolute difference (BRL),7.79
4,median difference as share of price + freight,0.106
5,share of HIGHER-payment orders paid in > 1 instalment,0.987
6,share of ALL orders paid in > 1 instalment,0.514
7,share of mismatches that used a voucher,0.02
8,orders with a gap in payment_sequential numbering,77
9,of which are mismatches,0


**Reading the payment evidence.** Almost all mismatches are orders where the customer paid **more** than price plus freight, and nearly all of those were paid in several instalments. That pattern is consistent with **instalment interest**, but the dataset does not record interest, so it is not proof. The few underpayments have no explanation in the data.

Gaps in the payment sequence numbers never coincide with a mismatch. They look like numbering artefacts rather than lost payments. Every value is kept as recorded.

In [12]:
zero_w_products = set(products.loc[products.product_weight_g == 0, "product_id"])
dims_missing_products = set(products.loc[products[["product_weight_g", "product_length_cm", "product_height_cm",
                                                    "product_width_cm"]].isna().any(axis=1), "product_id"])
flag_zero_weight = items.groupby("order_id").product_id.agg(lambda s: bool(set(s) & zero_w_products)).reindex(t.index, fill_value=False)
flag_dims_missing = t.weight_total_g.isna() | t.volume_total_cm3.isna()
items_with_missing_dims = items.product_id.isin(dims_missing_products)
assert flag_dims_missing.sum() == items.loc[items_with_missing_dims, "order_id"].nunique()

existing_flag_check = pd.DataFrame([
    {"flag (Phase 1 report figure)": "payment mismatch > 1 BRL (245)", "recomputed": int(mismatch.sum())},
    {"flag (Phase 1 report figure)": "no payment record (1)", "recomputed": int(t.payment_missing.sum())},
    {"flag (Phase 1 report figure)": "weight or size missing (16)", "recomputed": int(flag_dims_missing.sum())},
    {"flag (Phase 1 report figure)": "contains a zero-weight product (6)", "recomputed": int(flag_zero_weight.sum())},
    {"flag (Phase 1 report figure)": "comment of spaces only (26)", "recomputed": int((t.review_comment_message.notna() & (t.has_comment == 0)).sum())},
    {"flag (Phase 1 report figure)": "wait over 60 days (290)", "recomputed": int((t.wait_days > 60).sum())},
    {"flag (Phase 1 report figure)": "more than 30 days late (341)", "recomputed": int((t.delay_days > 30).sum())},
])
existing_flag_check

,flag (Phase 1 report figure),recomputed
0,payment mismatch > 1 BRL (245),245
1,no payment record (1),1
2,weight or size missing (16),16
3,contains a zero-weight product (6),6
4,comment of spaces only (26),26
5,wait over 60 days (290),290
6,more than 30 days late (341),341


## 5. Categorical data and encoding

Each check recomputes something independently from the source CSVs and compares it with what Phase 1 produced.

In [13]:
UF = {"AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA", "PB", "PR",
      "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"}   # Brazil's 26 states + Federal District
tr = src["translation"]
it = (items.merge(products, on="product_id", how="left", validate="many_to_one")
           .merge(tr, on="product_category_name", how="left", validate="many_to_one")
           .merge(src["sellers"], on="seller_id", how="left", validate="many_to_one"))
untranslated = sorted(it.loc[it.product_category_name.notna() & it.product_category_name_english.isna(),
                             "product_category_name"].unique())
unknown_orders = set(it.loc[it.product_category_name.isna(), "order_id"])

def multihot(df, col, prefix):
    m = pd.crosstab(df.order_id, df[col]).clip(upper=1).add_prefix(prefix).reindex(X.index, fill_value=0)
    return m[sorted(m.columns)]

seller_hot = multihot(it, "seller_state", "seller_state__")
pay_hot = multihot(pay, "payment_type", "pay__")
cust_hot = pd.get_dummies(t.customer_state, prefix="cust_state", dtype=int).reindex(X.index)
same_state_check = it.merge(t[["customer_state"]], left_on="order_id", right_index=True).assign(
    same=lambda d: d.seller_state == d.customer_state).groupby("order_id").same.all().reindex(t.index)

enc = {
    "customer states are valid Brazilian codes": set(src["customers"].customer_state) <= UF,
    "seller states are valid Brazilian codes": set(src["sellers"].seller_state) <= UF,
    "payment types are the 5 documented values": set(src["payments"].payment_type) <= {"credit_card", "boleto", "voucher", "debit_card", "not_defined"},
    "'not_defined' payment type absent from the sample": (pay.payment_type != "not_defined").all(),
    "only the 2 known categories lack an English name": untranslated == ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"],
    "no duplicate English category names": not tr.product_category_name_english.duplicated().any(),
    "cat__unknown = orders with an uncategorised item": set(X.index[X["cat__unknown"] == 1]) == unknown_orders,
    "dominant_category 'unknown' only where an item is uncategorised": set(t.index[t.dominant_category == "unknown"]) <= unknown_orders,
    "seller-state multi-hot matches source": (seller_hot.values == X[seller_hot.columns].values).all(),
    "payment-type multi-hot matches source": (pay_hot.values == X[pay_hot.columns].values).all(),
    "customer-state one-hot matches source": (cust_hot[sorted(cust_hot.columns)].values == X[sorted(cust_hot.columns)].values).all(),
    "exactly one customer state per order": (X.filter(like="cust_state_").sum(axis=1) == 1).all(),
    "category multi-hot row sums = n_categories": (X.filter(like="cat__").sum(axis=1).values == t.n_categories.values).all(),
    "seller-state multi-hot row sums = n_seller_states": (X.filter(like="seller_state__").sum(axis=1).values == t.n_seller_states.values).all(),
    "same_state = every seller in the customer's state": (same_state_check.astype(int).values == t.same_state.values).all(),
    "n_sellers matches source": (it.groupby("order_id").seller_id.nunique().reindex(t.index).values == t.n_sellers.values).all(),
    "price and freight totals match source": np.allclose(items.groupby("order_id").price.sum().reindex(t.index), t.items_price_total)
                                              and np.allclose(items.groupby("order_id").freight_value.sum().reindex(t.index), t.freight_total),
    "purchase month and weekday one-hot match": (pd.get_dummies(t.purchase_month.astype(str), prefix="month", dtype=int).values == X.filter(like="month_").values).all()
                                                and (X.filter(like="wday_").sum(axis=1) == 1).all(),
}
encoding_checks = pd.Series(enc, name="passed").to_frame()
encoding_checks.to_csv(OUT2 / "p2_encoding_checks.csv")
assert encoding_checks.passed.all(), encoding_checks[~encoding_checks.passed]
print(f"Multi-seller orders where every seller shares the customer's state: {int(((t.n_sellers > 1) & (t.same_state == 1)).sum())}")
encoding_checks

Multi-seller orders where every seller shares the customer's state: 338


,passed
customer states are valid Brazilian codes,True
seller states are valid Brazilian codes,True
payment types are the 5 documented values,True
'not_defined' payment type absent from the sample,True
only the 2 known categories lack an English name,True
no duplicate English category names,True
cat__unknown = orders with an uncategorised item,True
dominant_category 'unknown' only where an item is uncategorised,True
seller-state multi-hot matches source,True
payment-type multi-hot matches source,True


No rare category or state was dropped. All 74 category indicators and 22 seller-state indicators that occur in the sample are kept.

## 6. Audited table: every Phase 1 order plus documented flags

All 95,299 Phase 1 orders are kept here, with **no existing value changed**. The Phase 2 flags are added as new columns. `excluded_from_primary` and `exclusion_reason` mark the orders that the rule in section 7 removes from the primary sample, so that exclusion can always be traced and reversed.

In [14]:
audited = a.copy()
fi = audited.set_index("order_id").index
audited["has_any_text"] = ((audited.review_comment_message.fillna("").str.strip() != "") |
                           (audited.review_comment_title.fillna("").str.strip() != "")).astype(int)
audited["flag_title_only"] = ((audited.has_title == 1) & (audited.has_comment == 0)).astype(int)
audited["flag_whitespace_comment"] = (audited.review_comment_message.notna() & (audited.has_comment == 0)).astype(int)
audited["flag_category_unknown"] = audited.order_id.isin(unknown_orders).astype(int)
audited["flag_dims_missing"] = flag_dims_missing.reindex(fi).astype(int).values
audited["flag_zero_weight_product"] = flag_zero_weight.reindex(fi).astype(int).values
audited["flag_payment_mismatch"] = mismatch.reindex(fi).fillna(False).astype(int).values
audited["flag_installments_zero"] = (audited.max_installments == 0).astype(int)
audited["flag_wait_over_60"] = (audited.wait_days > 60).astype(int)
audited["flag_delay_over_30"] = (audited.delay_days > 30).astype(int)
audited["flag_promised_day_positive_delay"] = ((audited.delay_days > 0) & (audited.is_late == 0)).astype(int)
audited["flag_answered_over_60"] = (audited.review_lag_days > 60).astype(int)
audited["flag_review_before_purchase"] = (audited.review_answer_timestamp < audited.order_purchase_timestamp).astype(int)
audited["flag_lifecycle_inconsistent"] = ((audited.order_delivered_carrier_date < audited.order_purchase_timestamp) |
                                          (audited.order_delivered_customer_date < audited.order_delivered_carrier_date)).astype(int)
cust_counts = audited.customer_unique_id.map(audited.customer_unique_id.value_counts())
audited["flag_repeat_customer"] = (cust_counts > 1).astype(int)
audited["flag_sparse_period_2016"] = (audited.purchase_year == 2016).astype(int)
FLAG_COLS = [c for c in audited.columns if c not in a.columns]

# The Phase 2 chronological exclusion rule, recorded on the audited table
EXCLUSION_REASON = "review answered before the order was purchased (review_answer_timestamp < order_purchase_timestamp)"
audited["excluded_from_primary"] = audited.flag_review_before_purchase
audited["exclusion_reason"] = np.where(audited.excluded_from_primary == 1, EXCLUSION_REASON, "")

assert audited.order_id.is_unique and len(audited) == len(a) and (audited.order_id.values == a.order_id.values).all()
orig_equal = all(a[c].equals(audited[c]) for c in a.columns)
assert orig_equal, "an original column changed"

audited.to_parquet(OUT2 / "analysis_table_audited.parquet", index=False)
audited.to_csv(OUT2 / "analysis_table_audited.csv", index=False)
print(f"Audited table: {audited.shape[0]:,} rows x {audited.shape[1]} columns "
      f"({len(FLAG_COLS)} flag/derived columns + 2 exclusion columns added; original values unchanged)")

Audited table: 95,299 rows x 73 columns (16 flag/derived columns + 2 exclusion columns added; original values unchanged)


## 7. Chronological exclusion: reviews answered before purchase

**Exclusion rule:** exclude an order when `review_answer_timestamp < order_purchase_timestamp`. It is applied as **funnel step 6**, after the five Phase 1 steps, to the 95,299-order Phase 1 sample. The 4,142 Phase 1 exclusions are not counted again.

**Why.** The customer's answer is dated before the order existed, so it cannot be a reaction to that order. Two of the three reviews are also shared with another order. That makes it more likely the rating belongs to a different purchase, although the timestamps cannot tell us which one.

Applying this filter removes 3 chronological anomalies (all on time), leaving 95,296 orders (6,353 late and 88,943 on time or early). Each count is asserted below to verify consistency.

In [15]:
n_before = len(audited)
excluded = audited[audited.excluded_from_primary == 1]
final = audited[audited.excluded_from_primary == 0].drop(columns=["excluded_from_primary", "exclusion_reason"]).reset_index(drop=True)
# flag_repeat_customer depends on which orders are in the sample, so it is recomputed on the final sample
repeat_before = final.flag_repeat_customer.copy()
final["flag_repeat_customer"] = (final.customer_unique_id.map(final.customer_unique_id.value_counts()) > 1).astype(int)
print(f"flag_repeat_customer recomputed on the final sample: {int((repeat_before != final.flag_repeat_customer).sum())} order(s) "
      f"changed (a customer whose other order was excluded now has one order)")

# Verify expected sample counts after filtering
assert len(excluded) == 3, f"expected 3 exclusions, found {len(excluded)}"
assert int(excluded.is_late.sum()) == 0, "an excluded order is late"
assert len(final) == n_before - 3 == 95_296
assert int(final.is_late.sum()) == 6_353 == int(a.is_late.sum())
assert int((final.is_late == 0).sum()) == 88_943
assert final.order_id.is_unique and len(final) == final.order_id.nunique()
assert not (final.review_answer_timestamp < final.order_purchase_timestamp).any(), "a retained review precedes its purchase"
assert int(final.flag_review_before_purchase.sum()) == 0
assert (final.order_id.values == a.order_id[a.order_id.isin(final.order_id)].values).all(), "order changed"
assert all(final[c].equals(a.loc[a.order_id.isin(final.order_id), c].reset_index(drop=True)) for c in a.columns), "an original value changed"

excluded_out = excluded[["order_id", "review_id", "order_purchase_timestamp", "review_creation_date", "review_answer_timestamp",
                         "review_shared_across_orders", "is_late", "review_score", "exclusion_reason"]].copy()
excluded_out["orders_sharing_this_review"] = excluded_out.review_id.map(lambda r: len(shared_with[r]))
excluded_out.to_csv(OUT2 / "p2_excluded_orders.csv", index=False)

funnel_final = pd.concat([funnel.assign(phase="Phase 1"), pd.DataFrame([{
    "step": "6. review answered on or after purchase", "rows_after": len(final), "removed": len(excluded),
    "reason": "review answered before the order was purchased (Phase 2 chronological rule)", "phase": "Phase 2"}])],
    ignore_index=True)
assert funnel_final.rows_after.iloc[-2] == n_before and funnel_final.rows_after.iloc[-1] == len(final)
funnel_final.to_csv(OUT2 / "p2_filter_funnel_final.csv", index=False)
print(f"Step 6: {n_before:,} -> {len(final):,} (removed {len(excluded)}); late {int(final.is_late.sum()):,}, "
      f"on time or early {int((final.is_late == 0).sum()):,}")
display(funnel_final)
excluded_out

flag_repeat_customer recomputed on the final sample: 1 order(s) changed (a customer whose other order was excluded now has one order)
Step 6: 95,299 -> 95,296 (removed 3); late 6,353, on time or early 88,943


,step,rows_after,removed,reason,phase
0,0. all orders,99441,0,NaN,Phase 1
1,1. status == delivered,96478,2963,"not delivered (shipped, canceled, unavailable, ...)",Phase 1
2,2. delivery timestamp present,96470,8,lateness and waiting time cannot be computed,Phase 1
3,3. estimated date present,96470,0,lateness cannot be computed,Phase 1
4,4. at least one review,95824,646,"no review, so no outcome",Phase 1
5,5. exactly one review,95299,525,several reviews: ambiguous outcome (D2),Phase 1
6,6. review answered on or after purchase,95296,3,review answered before the order was purchased (Phase 2 chronological rule),Phase 2


,order_id,review_id,order_purchase_timestamp,review_creation_date,review_answer_timestamp,review_shared_across_orders,is_late,review_score,exclusion_reason,orders_sharing_this_review
31108,ebc94658c583ab37ad4f8e9091c4bef2,d25026282e14fc531f0711246e121cc2,2018-04-14 08:22:43,2018-03-30,2018-04-02 00:32:35,1,0,2,review answered before the order was purchased (review_answer_timestamp < or...,2
47851,4bb9c2002502ca416276dc1ff5efb1b3,0857872f03ef22e3dbe8391c4dd331f6,2018-06-23 14:53:03,2018-05-18,2018-05-21 18:15:06,0,0,1,review answered before the order was purchased (review_answer_timestamp < or...,1
66814,4a62fb19d5fa08fb514619dfcc617b3d,a935b3d67f406bac4853776d80a05220,2018-05-09 19:23:46,2018-05-05,2018-05-08 02:06:50,1,0,3,review answered before the order was purchased (review_answer_timestamp < or...,2


### Dependent tables for the final sample

The outcomes table and the encoded matrix are rebuilt for the 95,296 orders, in the same `order_id` order, rather than leaving them at the old size.

The matrix is re-imputed with medians computed on the **final** sample, constant columns are checked again, and the size requirement is recomputed. The attribute count excludes `order_id`, which is the index and not a feature.

In [16]:
ids = final.order_id
outcomes_final = outcomes.loc[ids]
assert (outcomes_final.index.values == ids.values).all() and not outcomes_final.index.duplicated().any()

X_final = X.loc[ids].copy()
reimputed = {}
for c, n_imp in p1_summary["imputed_values"].items():
    missing_rows = final[c].isna().values
    med = final[c].median()
    X_final.loc[missing_rows, c] = med
    reimputed[c] = {"filled": int(missing_rows.sum()), "median_final_sample": float(med)}
const_final = [c for c in X_final.columns if X_final[c].nunique() <= 1]
X_final = X_final.drop(columns=const_final)
assert not X_final.isna().any().any(), "X_final has missing values"
assert (X_final.index.values == ids.values).all() and X_final.index.name == "order_id"
assert "order_id" not in X_final.columns

n_inst, n_attr = X_final.shape
product = n_inst * n_attr
assert product >= 3_000_000
outcomes_final.to_csv(OUT2 / "outcomes_final.csv")
X_final.to_parquet(OUT2 / "X_encoded_final.parquet")
X_final.to_csv(OUT2 / "X_encoded_final.csv")
final.to_parquet(OUT2 / "analysis_table_final.parquet", index=False)
final.to_csv(OUT2 / "analysis_table_final.csv", index=False)

print("Re-imputed with final-sample medians:", reimputed)
print("Constant columns dropped:", const_final)
print(f"Encoded matrix for the final sample: {n_inst:,} instances x {n_attr} attributes = {product:,} "
      f"(>= 3,000,000: {product >= 3_000_000}; order_id is the index, not counted)")

Re-imputed with final-sample medians: {'payment_total': {'filled': 1, 'median_final_sample': 105.28}, 'n_payment_records': {'filled': 1, 'median_final_sample': 1.0}, 'max_installments': {'filled': 1, 'median_final_sample': 2.0}, 'weight_total_g': {'filled': 16, 'median_final_sample': 750.0}, 'volume_total_cm3': {'filled': 16, 'median_final_sample': 7220.0}}
Constant columns dropped: []
Encoded matrix for the final sample: 95,296 instances x 167 attributes = 15,914,432 (>= 3,000,000: True; order_id is the index, not counted)


In [17]:
before_after = pd.DataFrame([
    {"measure": "rows", "Phase 1 table": len(a), "audited (Phase 2)": len(audited), "final primary (Phase 2)": len(final)},
    {"measure": "unique order_id", "Phase 1 table": a.order_id.nunique(), "audited (Phase 2)": audited.order_id.nunique(), "final primary (Phase 2)": final.order_id.nunique()},
    {"measure": "columns", "Phase 1 table": a.shape[1], "audited (Phase 2)": audited.shape[1], "final primary (Phase 2)": final.shape[1]},
    {"measure": "late orders", "Phase 1 table": int(a.is_late.sum()), "audited (Phase 2)": int(audited.is_late.sum()), "final primary (Phase 2)": int(final.is_late.sum())},
    {"measure": "on time or early", "Phase 1 table": int((a.is_late == 0).sum()), "audited (Phase 2)": int((audited.is_late == 0).sum()), "final primary (Phase 2)": int((final.is_late == 0).sum())},
    {"measure": "orders excluded by the Phase 2 rule", "Phase 1 table": 0, "audited (Phase 2)": "3 marked", "final primary (Phase 2)": len(excluded)},
    {"measure": "missing cells in the original 55 columns", "Phase 1 table": int(a.isna().sum().sum()),
     "audited (Phase 2)": int(audited[a.columns].isna().sum().sum()), "final primary (Phase 2)": int(final[a.columns].isna().sum().sum())},
    {"measure": "original values changed", "Phase 1 table": 0, "audited (Phase 2)": 0 if orig_equal else "CHANGED", "final primary (Phase 2)": 0},
    {"measure": "outcomes table rows", "Phase 1 table": len(outcomes), "audited (Phase 2)": "-", "final primary (Phase 2)": len(outcomes_final)},
    {"measure": "encoded matrix (instances x attributes)", "Phase 1 table": f"{X.shape[0]} x {X.shape[1]}", "audited (Phase 2)": "-",
     "final primary (Phase 2)": f"{n_inst} x {n_attr}"},
    {"measure": "instances x attributes", "Phase 1 table": X.shape[0] * X.shape[1], "audited (Phase 2)": "-", "final primary (Phase 2)": product},
])
before_after.to_csv(OUT2 / "p2_before_after.csv", index=False)
before_after

,measure,Phase 1 table,audited (Phase 2),final primary (Phase 2)
0,rows,95299,95299,95296
1,unique order_id,95299,95299,95296
2,columns,55,73,71
3,late orders,6353,6353,6353
4,on time or early,88946,88946,88943
5,orders excluded by the Phase 2 rule,0,3 marked,3
6,missing cells in the original 55 columns,140755,140755,140753
7,original values changed,0,0,0
8,outcomes table rows,95299,-,95296
9,encoded matrix (instances x attributes),95299 x 167,-,95296 x 167


## 8. Quality flags, missingness, cleaning log and reconciliation (final sample)

Flag counts are reported for the **final** sample, with the audited-sample count beside them. The missingness summary gains rows for the final table. Every count in the cleaning log is read from the data.

In [18]:
FLAG_INFO = {
    "review_before_delivery": ("Phase 1", "review answered before the parcel arrived (D4)", "retained; Phase 4 sensitivity"),
    "survey_sent_before_delivery": ("Phase 1", "survey created before the delivery day", "retained; descriptive"),
    "review_shared_across_orders": ("Phase 1", "review record attached to another order (D5)", "retained; Phase 4 sensitivity"),
    "payment_missing": ("Phase 1", "no payment record", "retained; median in X only"),
    "has_any_text": ("Phase 2", "message or title contains text", "derived feature; alternative to has_comment"),
    "flag_title_only": ("Phase 2", "title but no message", "retained; has_comment = 0"),
    "flag_whitespace_comment": ("Phase 2", "message of spaces only", "retained; has_comment = 0; text unchanged"),
    "flag_category_unknown": ("Phase 2", "order has an item with no category", "retained; cat__unknown"),
    "flag_dims_missing": ("Phase 2", "weight or size missing for an item", "retained; NaN in table, median in X"),
    "flag_zero_weight_product": ("Phase 2", "order contains a product recorded as 0 g", "retained as recorded"),
    "flag_payment_mismatch": ("Phase 2", "payment differs from price + freight by > 1 BRL", "retained; likely instalment interest, unproven"),
    "flag_installments_zero": ("Phase 2", "card payment recorded with 0 instalments", "retained as recorded"),
    "flag_wait_over_60": ("Phase 2", "waiting time over 60 days", "retained; possible long deliveries"),
    "flag_delay_over_30": ("Phase 2", "more than 30 days late", "retained"),
    "flag_promised_day_positive_delay": ("Phase 2", "delay_days > 0 on the promised day, is_late = 0", "correct under D3"),
    "flag_answered_over_60": ("Phase 2", "review answered over 60 days after delivery", "retained"),
    "flag_review_before_purchase": ("Phase 2", "review answered before the order was purchased", "EXCLUDED from the primary sample (step 6)"),
    "flag_lifecycle_inconsistent": ("Phase 2", "carrier date before purchase or after customer delivery", "retained; columns unused"),
    "flag_repeat_customer": ("Phase 2", "customer has more than one order in the sample (recomputed for the final sample)", "retained; clustering for Phase 4"),
    "flag_sparse_period_2016": ("Phase 2", "purchased in 2016 (only a few months, thin data)", "retained; treat with care in time plots"),
}
qf = pd.DataFrame([{"flag": f, "defined_in": d, "meaning": m, "orders_final_sample": int(final[f].sum()),
                    "pct_final_sample": round(100 * final[f].mean(), 2), "orders_audited_sample": int(audited[f].sum()),
                    "handling": h} for f, (d, m, h) in FLAG_INFO.items()])
qf.to_csv(OUT2 / "p2_quality_flags.csv", index=False)
qf

,flag,defined_in,meaning,orders_final_sample,pct_final_sample,orders_audited_sample,handling
0,review_before_delivery,Phase 1,review answered before the parcel arrived (D4),4624,4.85,4627,retained; Phase 4 sensitivity
1,survey_sent_before_delivery,Phase 1,survey created before the delivery day,4937,5.18,4940,retained; descriptive
2,review_shared_across_orders,Phase 1,review record attached to another order (D5),867,0.91,869,retained; Phase 4 sensitivity
3,payment_missing,Phase 1,no payment record,1,0.00,1,retained; median in X only
4,has_any_text,Phase 2,message or title contains text,40397,42.39,40400,derived feature; alternative to has_comment
5,flag_title_only,Phase 2,title but no message,1701,1.78,1702,retained; has_comment = 0
6,flag_whitespace_comment,Phase 2,message of spaces only,26,0.03,26,retained; has_comment = 0; text unchanged
7,flag_category_unknown,Phase 2,order has an item with no category,1374,1.44,1374,retained; cat__unknown
8,flag_dims_missing,Phase 2,weight or size missing for an item,16,0.02,16,"retained; NaN in table, median in X"
9,flag_zero_weight_product,Phase 2,order contains a product recorded as 0 g,6,0.01,6,retained as recorded


In [19]:
# Missingness summary: add the final primary table to the source and Phase 1 rows written in section 2
final_rows = []
for c in final.columns:
    n = int(final[c].isna().sum())
    treat, why = TREAT.get(("analysis", c), ("none needed", "no missing values") if n == 0 else ("REVIEW", "no treatment defined"))
    final_rows.append({"table": "analysis_final", "column": c, "dtype": str(final[c].dtype), "rows": len(final), "missing": n,
                       "missing_pct": round(100 * n / len(final), 2), "treatment": treat, "justification": why})
missingness_all = pd.concat([missingness.replace({"table": {"analysis": "analysis_phase1"}}), pd.DataFrame(final_rows)],
                            ignore_index=True)
assert (missingness_all.treatment != "REVIEW").all()
missingness_all.to_csv(OUT2 / "p2_missingness_summary.csv", index=False)
missingness_all[(missingness_all.table == "analysis_final") & (missingness_all.missing > 0)]

,table,column,dtype,rows,missing,missing_pct,treatment,justification
113,analysis_final,order_approved_at,datetime64[us],95296,14,0.01,kept missing,not used by any feature
114,analysis_final,order_delivered_carrier_date,datetime64[us],95296,1,0.00,kept missing,not used by any feature
123,analysis_final,payment_types,str,95296,1,0.00,kept missing; all pay__ indicators 0 in X,same order as payment_total
141,analysis_final,payment_total,float64,95296,1,0.00,missing in table; median in X only,one order has no payment record in the source
142,analysis_final,n_payment_records,float64,95296,1,0.00,missing in table; median in X only,same order as payment_total
143,analysis_final,max_installments,float64,95296,1,0.00,missing in table; median in X only,same order as payment_total
144,analysis_final,weight_total_g,float64,95296,16,0.02,missing in table; median in X only,an order with an item of unknown weight has unknown total weight
145,analysis_final,volume_total_cm3,float64,95296,16,0.02,missing in table; median in X only,same reason as weight
160,analysis_final,review_comment_title,str,95296,84128,88.28,kept missing; has_title = 0,text left unchanged
161,analysis_final,review_comment_message,str,95296,56574,59.37,kept missing; has_comment = 0,text left unchanged for Phase 3


In [20]:
F = final
fun = funnel_final.set_index("step").removed
p1_rev_old = int(((audited.review_creation_date < audited.order_delivered_customer_date.dt.normalize()) != (audited.review_before_delivery == 1)).sum())
n_untranslated_items = int((it.product_category_name.isin(untranslated)).sum())
undocumented = ["seller_states", "payment_types", "payment_missing"]

L = []
def log(cat, src_, problem, n, decision, reason, changed, impact, limitation, sens):
    L.append({"category": cat, "source_or_feature": src_, "problem": problem, "records_affected": int(n), "decision": decision,
              "reason": reason, "changed_or_removed": changed, "impact_on_primary_sample": impact,
              "limitation_or_bias": limitation, "sensitivity_recommended": sens})

# Corrected
log("Corrected", "review_before_delivery, review_lag_days", "Comparing survey creation date vs review answer timestamp",
    p1_rev_old, "computed from review_answer_timestamp vs delivery timestamp",
    "only the answer time shows whether the customer had the parcel", "feature values changed; no rows",
    f"none (classification of {p1_rev_old:,} orders changed)", "none", "no")
log("Corrected", "products.product_category_name -> translation", "2 categories have no English name", n_untranslated_items,
    "documented English names added (pc_gamer, portable_kitchen_food_preparers)", "avoid losing real categories",
    "translation added; no rows", "none", "translation is ours, not Olist's", "no")
log("Corrected", "04_feature_dictionary.csv", f"{len(undocumented)} analysis columns undocumented: {', '.join(undocumented)}",
    len(undocumented), "complete dictionary written (p2_feature_dictionary.csv)", "every column must be explainable",
    "documentation only", "none", "none", "no")

# Excluded: Phase 1 funnel rules and Phase 2 chronological rule
for step, why in [("1. status == delivered", "not delivered"), ("2. delivery timestamp present", "delivered but no arrival time"),
                  ("4. at least one review", "no review, so no outcome"), ("5. exactly one review", "several reviews, ambiguous outcome (D2)")]:
    log("Excluded", "orders / reviews", why, fun[step], f"removed by Phase 1 funnel step '{step}'", "eligibility rule",
        "removed", f"-{int(fun[step]):,} orders", "excluded orders may differ systematically (e.g. canceled orders)",
        "yes (keep-latest rule)" if step.startswith("5") else "no")
log("Excluded", "flag_review_before_purchase", "review answered before the order was purchased", fun["6. review answered on or after purchase"],
    "removed by Phase 2 funnel step 6; kept and marked in analysis_table_audited",
    "the rating cannot be about an order that did not yet exist; 2 of 3 reviews are shared with another order",
    "removed from the primary sample; source and audited table unchanged",
    f"-{len(excluded)} orders ({n_before:,} -> {len(final):,}); all on time, late count unchanged",
    "negligible: 3 of 88,946 on-time orders", "no")

# Retained and flagged (counts on the final sample)
RF = [
    ("review_before_delivery", "review answered before actual delivery", "kept; D4", "experience while waiting is part of lateness", "mixes 'waiting' and 'received' ratings", "yes: exclude them"),
    ("review_shared_across_orders", "one review record attached to several orders", "kept; D5", "outcome is clear", "same rating counted for several orders: weakens independence", "yes: exclude them"),
    ("flag_payment_mismatch", "payment differs from price + freight by > 1 BRL", "kept as recorded", "pattern fits instalment interest; not provable", "payment_total is not a clean price measure", "no"),
    ("flag_installments_zero", "card payment with 0 instalments", "kept as recorded", "invalid value but unused by the question", "none for the question", "no"),
    ("flag_zero_weight_product", "product weight recorded as 0 g", "kept as recorded", "cannot know the true weight", "weight slightly understated", "no"),
    ("flag_wait_over_60", "waiting time over 60 days", "kept", "possible long deliveries; no arbitrary cut", "heavy right tail: prefer robust statistics", "yes: robust tests"),
    ("flag_delay_over_30", "more than 30 days late", "kept", "severe but possible delays; no evidence of error", "heavy tail", "yes: robust tests"),
    ("flag_promised_day_positive_delay", "delay_days > 0 but is_late = 0", "kept; correct under D3", "estimate stored as midnight", "do not read small positive delay as late", "yes: timestamp rule"),
    ("flag_answered_over_60", "answered over 60 days after delivery", "kept", "late answers are still answers", "memory may fade", "no"),
    ("flag_lifecycle_inconsistent", "carrier date before purchase or after customer delivery", "kept; columns unused", "affects only unused timestamps", "none for the question", "no"),
    ("flag_title_only", "title but no message", "kept; has_any_text added", "a title is still written feedback", "has_comment undercounts writing", "yes: has_any_text"),
    ("flag_repeat_customer", "customer with several eligible orders", "kept", "real orders", "clustered observations", "yes: one order per customer"),
    ("flag_sparse_period_2016", "2016 purchases (thin data)", "kept", "real orders", "unstable monthly estimates for 2016", "no (care in Phase 3)"),
]
for col, prob, dec, why, lim, sens in RF:
    log("Retained and flagged", col, prob, F[col].sum(), dec, why, "no", "none", lim, sens)
log("Retained and flagged", "payments.payment_sequential", "gap in payment sequence numbers", seq_gap.reindex(F.order_id).sum(),
    "kept", "no gap coincides with a payment mismatch", "no", "none", "none found", "no")
log("Retained and flagged", "freight_ratio", "freight greater than item price", (F.freight_ratio > 1).sum(),
    "kept", "cheap items with standard freight are real", "no", "none", "skewed ratio: prefer robust statistics", "no")

# Missing-value handling (final sample)
MV = [
    ("review_comment_message", "no comment message (null)", F.review_comment_message.isna().sum(), "kept missing; has_comment = 0", "rating without writing is allowed"),
    ("review_comment_message", "comment of spaces only", F.flag_whitespace_comment.sum(), "text unchanged; has_comment = 0", "no content"),
    ("review_comment_title", "no title", F.review_comment_title.isna().sum(), "kept missing; has_title = 0", "titles are optional"),
    ("product_category_name", "order contains an uncategorised item", F.flag_category_unknown.sum(), "'unknown' at order level; flag", "no category invented"),
    ("weight_total_g, volume_total_cm3", "item weight or size missing", F.flag_dims_missing.sum(), "NaN in table; median in X only", "total unknown if any item unknown"),
    ("payment_total, n_payment_records, max_installments, payment_types", "no payment record", F.payment_missing.sum(), "NaN in table; median in X; pay__ = 0", "source has no record"),
    ("order_approved_at", "approval time missing", F.order_approved_at.isna().sum(), "kept missing", "column unused"),
    ("order_delivered_carrier_date", "carrier time missing", F.order_delivered_carrier_date.isna().sum(), "kept missing", "column unused"),
]
for col, prob, n, dec, why in MV:
    log("Missing-value handling", col, prob, n, dec, why, "no (imputation only in X)" if "median" in dec else "no", "none",
        "imputed values carry no information" if "median" in dec else "none", "no")

cleaning_log = pd.DataFrame(L)
cleaning_log.insert(0, "issue_id", [f"{c[0]}{i:02d}" for i, c in enumerate(cleaning_log.category, 1)])
cleaning_log.to_csv(OUT2 / "p2_cleaning_log.csv", index=False)
cleaning_log[["issue_id", "category", "source_or_feature", "problem", "records_affected", "decision", "sensitivity_recommended"]]

,issue_id,category,source_or_feature,problem,records_affected,decision,sensitivity_recommended
0,C01,Corrected,"review_before_delivery, review_lag_days",Comparing survey creation date vs review answer timestamp,713,computed from review_answer_timestamp vs delivery timestamp,no
1,C02,Corrected,products.product_category_name -> translation,2 categories have no English name,22,"documented English names added (pc_gamer, portable_kitchen_food_preparers)",no
2,C03,Corrected,04_feature_dictionary.csv,"3 analysis columns undocumented: seller_states, payment_types, payment_missing",3,complete dictionary written (p2_feature_dictionary.csv),no
3,E04,Excluded,orders / reviews,not delivered,2963,removed by Phase 1 funnel step '1. status == delivered',no
4,E05,Excluded,orders / reviews,delivered but no arrival time,8,removed by Phase 1 funnel step '2. delivery timestamp present',no
5,E06,Excluded,orders / reviews,"no review, so no outcome",646,removed by Phase 1 funnel step '4. at least one review',no
6,E07,Excluded,orders / reviews,"several reviews, ambiguous outcome (D2)",525,removed by Phase 1 funnel step '5. exactly one review',yes (keep-latest rule)
7,E08,Excluded,flag_review_before_purchase,review answered before the order was purchased,3,removed by Phase 2 funnel step 6; kept and marked in ...,no
8,R09,Retained and flagged,review_before_delivery,review answered before actual delivery,4624,kept; D4,yes: exclude them
9,R10,Retained and flagged,review_shared_across_orders,one review record attached to several orders,867,kept; D5,yes: exclude them


In [21]:
excluded_total = int(cleaning_log.loc[cleaning_log.category == "Excluded", "records_affected"].sum())
recon = {
    "excluded in log + final rows = source orders": excluded_total + len(final) == len(src["orders"]),
    "funnel removals (Phase 1 + step 6) = excluded in log": int(funnel_final.removed.sum()) == excluded_total,
    "Phase 1 exclusions not double-counted (4,142 + 3)": int(funnel.removed.sum()) + len(excluded) == excluded_total,
    "step 6 removed exactly 3 orders": int(fun["6. review answered on or after purchase"]) == 3,
    "final rows = unique order_id": len(final) == final.order_id.nunique(),
    "final rows = 95,296": len(final) == 95_296,
    "late orders = 6,353; on time or early = 88,943": int(final.is_late.sum()) == 6_353 and int((final.is_late == 0).sum()) == 88_943,
    "no retained review answered before purchase": not (final.review_answer_timestamp < final.order_purchase_timestamp).any(),
    "every flag's log count = its sum in the final table": all(
        int(final[r.source_or_feature].sum()) == r.records_affected
        for r in cleaning_log.itertuples() if r.source_or_feature in FLAG_INFO and r.category == "Retained and flagged"),
    "quality-flag CSV counts = final and audited table sums": all(
        int(final[r.flag].sum()) == r.orders_final_sample and int(audited[r.flag].sum()) == r.orders_audited_sample for r in qf.itertuples()),
    "audited table marks exactly the excluded orders": set(audited.order_id[audited.excluded_from_primary == 1]) == set(excluded.order_id),
    "audited minus excluded = final (same order)": (audited.order_id[audited.excluded_from_primary == 0].values == final.order_id.values).all(),
    "final table, outcomes_final and X_encoded_final share order_id order": (
        (final.order_id.values == outcomes_final.index.values).all() and (final.order_id.values == X_final.index.values).all()),
    "X_encoded_final has no missing values": not X_final.isna().any().any(),
    "X_encoded_final instances x attributes >= 3,000,000": product >= 3_000_000,
    "Phase 1 outputs untouched (still 95,299 rows)": len(pd.read_parquet(P1_DIR / "analysis_table.parquet")) == 95_299,
}
recon = pd.Series(recon, name="passed").to_frame()
recon.to_csv(OUT2 / "p2_reconciliation.csv")
assert recon.passed.all(), recon[~recon.passed]
print(f"{int(funnel.removed.sum()):,} Phase 1 exclusions + {len(excluded)} Phase 2 exclusions + {len(final):,} kept = {len(src['orders']):,} source orders")
recon

4,142 Phase 1 exclusions + 3 Phase 2 exclusions + 95,296 kept = 99,441 source orders


,passed
excluded in log + final rows = source orders,True
funnel removals (Phase 1 + step 6) = excluded in log,True
"Phase 1 exclusions not double-counted (4,142 + 3)",True
step 6 removed exactly 3 orders,True
final rows = unique order_id,True
"final rows = 95,296",True
"late orders = 6,353; on time or early = 88,943",True
no retained review answered before purchase,True
every flag's log count = its sum in the final table,True
quality-flag CSV counts = final and audited table sums,True


## 9. Complete feature dictionary

In [22]:
D_ = {
 "order_id": ("identifier", "order key (one row per order)"), "customer_id": ("identifier", "per-order customer key"),
 "customer_unique_id": ("identifier", "real-person key; repeats across a customer's orders"), "review_id": ("identifier", "review key; may be shared across orders (D5)"),
 "order_status": ("raw", "always 'delivered' in this sample"),
 "order_purchase_timestamp": ("raw timestamp", "when the order was placed"), "order_approved_at": ("raw timestamp", "payment approval time; unused"),
 "order_delivered_carrier_date": ("raw timestamp", "hand-over to carrier; unused"), "order_delivered_customer_date": ("raw timestamp", "actual arrival at the customer"),
 "order_estimated_delivery_date": ("raw timestamp", "promised date (midnight, no time of day)"),
 "review_creation_date": ("raw timestamp", "when Olist created/sent the survey (not the answer)"), "review_answer_timestamp": ("raw timestamp", "when the customer submitted the review"),
 "customer_city": ("raw", "customer city (not encoded: ~4,100 levels)"), "customer_state": ("location", "customer state (27); one-hot in X"),
 "seller_states": ("location", "'|'-joined sorted set of seller states in the order"), "dominant_category": ("product", "category with the largest share of item price; EDA label only"),
 "payment_types": ("payment", "'|'-joined set of payment types used; missing if no payment record"),
 "purchase_yearmonth": ("time", "YYYY-MM of purchase"), "purchase_month": ("time", "1-12; one-hot in X"), "purchase_weekday": ("time", "Mon..Sun; one-hot in X"),
 "purchase_hour": ("time", "0-23"), "purchase_year": ("time", "2016-2018"), "is_weekend": ("time", "1 if purchased Sat/Sun"),
 "delay_calendar_days": ("delivery", "delivery date - estimated date, whole days; > 0 means late"), "payment_missing": ("payment", "1 if the order has no payment record"),
 "is_late": ("delivery", "1 if delivered on a later calendar day than promised (D3)"), "delay_days": ("delivery", "delivery timestamp - estimated date, days; negative = early"),
 "wait_days": ("delivery", "delivery - purchase timestamp, days"), "promised_days": ("delivery", "estimated date - purchase timestamp, days"),
 "items_price_total": ("order", "sum of item prices, BRL"), "freight_total": ("order", "sum of freight, BRL"), "freight_ratio": ("order", "freight_total / items_price_total"),
 "n_items": ("order", "item rows (units)"), "n_products": ("order", "distinct products"), "n_sellers": ("order", "distinct sellers"),
 "n_categories": ("order", "distinct categories ('unknown' counts as one)"), "n_seller_states": ("order", "distinct seller states"),
 "payment_total": ("payment", "sum of payment values, BRL"), "n_payment_records": ("payment", "payment records"), "max_installments": ("payment", "largest instalment count"),
 "weight_total_g": ("product", "sum of item weights, g; missing if any item unknown"), "volume_total_cm3": ("product", "sum of item L x H x W, cm3; missing if any unknown"),
 "same_state": ("location", "1 if every seller is in the customer's state"),
 "review_score": ("outcome", "1-5 stars"), "low_score": ("outcome", "1 if score is 1 or 2"),
 "has_comment": ("outcome/text", "1 if the message has non-blank text"), "comment_len_chars": ("outcome/text", "characters in the trimmed message; 0 if none"),
 "has_title": ("outcome/text", "1 if the title has non-blank text"),
 "review_before_delivery": ("review timing", "1 if review_answer_timestamp < order_delivered_customer_date (D4)"),
 "review_lag_days": ("review timing", "answer timestamp - delivery timestamp, days"),
 "survey_sent_before_delivery": ("survey timing", "1 if the survey was created before the delivery day"),
 "survey_lag_days": ("survey timing", "survey date - delivery date, whole days"),
 "review_shared_across_orders": ("data quality", "1 if the review_id is attached to another order (D5)"),
 "review_comment_title": ("raw text", "review title, unchanged (Portuguese)"), "review_comment_message": ("raw text", "review message, unchanged (Portuguese)"),
}
for f, (d, m, h) in FLAG_INFO.items():
    if f not in D_:
        D_[f] = ("quality flag" if f.startswith("flag_") else "outcome/text", f"{m} ({d})")
D_["flag_review_before_purchase"] = ("quality flag", "1 if the review was answered before purchase (Phase 2); always 0 in the final table because these orders are excluded")
AUDIT_ONLY = {"excluded_from_primary": ("audit", "1 if the order is excluded from the primary sample by the Phase 2 rule"),
              "exclusion_reason": ("audit", "why the order is excluded; empty if retained")}
fam = {"cat__<category> (X)": ("product", "multi-hot: order contains this English category (74)"),
       "seller_state__<UF> (X)": ("location", "multi-hot: a seller of the order is in this state (22)"),
       "pay__<type> (X)": ("payment", "multi-hot: order used this payment type (4)"),
       "cust_state_<UF> (X)": ("location", "one-hot customer state (27)"),
       "month_<m> (X)": ("time", "one-hot purchase month (12)"), "wday_<day> (X)": ("time", "one-hot purchase weekday (7)")}
p2_dict = pd.DataFrame([{"feature": k, "group": g, "meaning": m, "in": "analysis_table_final and analysis_table_audited"} for k, (g, m) in D_.items()] +
                       [{"feature": k, "group": g, "meaning": m, "in": "analysis_table_audited only"} for k, (g, m) in AUDIT_ONLY.items()] +
                       [{"feature": k, "group": g, "meaning": m, "in": "X_encoded_final (and Phase 1 X_encoded)"} for k, (g, m) in fam.items()])
documented = set(D_) | set(AUDIT_ONLY)
assert not [c for c in audited.columns if c not in documented], "undocumented column"
assert not [c for c in final.columns if c not in D_], "undocumented column"
p2_dict.to_csv(OUT2 / "p2_feature_dictionary.csv", index=False)
print(f"Documented: {final.shape[1]} final-table columns, {len(AUDIT_ONLY)} audit-only columns, {len(fam)} encoded families.")

Documented: 71 final-table columns, 2 audit-only columns, 6 encoded families.


## 10. Summary

In [23]:
summary2 = {
    "final_primary_table": str(OUT2 / "analysis_table_final.parquet"),
    "final_rows": int(len(final)), "final_columns": int(final.shape[1]),
    "late_orders": int(final.is_late.sum()), "on_time_or_early": int((final.is_late == 0).sum()),
    "phase1_rows": int(len(a)), "phase2_excluded": int(len(excluded)),
    "phase2_exclusion_rule": EXCLUSION_REASON,
    "excluded_order_ids": excluded.order_id.tolist(),
    "audited_table": str(OUT2 / "analysis_table_audited.parquet"),
    "X_encoded_final": {"instances": int(n_inst), "attributes": int(n_attr), "product": int(product), "reimputed": reimputed,
                        "constant_columns_dropped": const_final},
    "columns_added_in_phase2": FLAG_COLS,
    "original_values_changed": not orig_equal,
    "pct_without_comment_final": round(float(100 * (final.has_comment == 0).mean()), 2),
    "cleaning_log_entries_by_category": cleaning_log.category.value_counts().to_dict(),
    "encoding_checks_passed": bool(encoding_checks.passed.all()), "reconciliation_passed": bool(recon.passed.all()),
}
json.dump(summary2, open(OUT2 / "p2_summary.json", "w"), indent=2)
for k, v in summary2.items():
    print(f"{k:34s} {v}")

final_primary_table                outputs\phase2\analysis_table_final.parquet
final_rows                         95296
final_columns                      71
late_orders                        6353
on_time_or_early                   88943
phase1_rows                        95299
phase2_excluded                    3
phase2_exclusion_rule              review answered before the order was purchased (review_answer_timestamp < order_purchase_timestamp)
excluded_order_ids                 ['ebc94658c583ab37ad4f8e9091c4bef2', '4bb9c2002502ca416276dc1ff5efb1b3', '4a62fb19d5fa08fb514619dfcc617b3d']
audited_table                      outputs\phase2\analysis_table_audited.parquet
X_encoded_final                    {'instances': 95296, 'attributes': 167, 'product': 15914432, 'reimputed': {'payment_total': {'filled': 1, 'median_final_sample': 105.28}, 'n_payment_records': {'filled': 1, 'median_final_sample': 1.0}, 'max_installments': {'filled': 1, 'median_final_sample': 2.0}, 'weight_total_g': {'fil

### Methodological Notes for Data Quality Audit

**Why did you exclude three orders?** Each one's review was answered *before the order was placed*. A customer cannot rate a delivery that does not exist yet, so the score is not an outcome of that order. Two of the three reviews are also attached to another order, which suggests the customer was rating a different purchase. We cannot prove which purchase it was, and we do not need to: it is enough that the rating cannot be about this order.

**Did it change the results?** Hardly. All three orders were on time, so the late group (6,353) is untouched, and the on-time group drops from 88,946 to 88,943. The orders are not deleted anywhere: they stay in the source CSVs, in the Phase 1 outputs and in `analysis_table_audited`, marked with `excluded_from_primary` and `exclusion_reason`. The decision is fully traceable and can be reversed.

**Why didn't you remove outliers?** An unusual value is not the same as a wrong value. A 200-day delivery or a BRL 13,000 order really happened, and cutting them at an arbitrary threshold would change the answer to the research question. We flag them, and we plan robust tests that do not let a few extreme values dominate.

**How did you handle missing values?** We kept them in the analysis table, so the truth is visible. Only the encoded matrix fills gaps, with the median of the final sample, because that matrix must have no gaps. Missing categories become an explicit `unknown` category; we never invent one.

**Why is the no-comment rate about 59.4% and not 58.7%?** 58.7% is computed over all 99,224 review rows, and our sample is the eligible orders. A percentage always belongs to the population it was computed on.